# Lab 3: Explainable AI สำหรับ Deep Learning ด้วย Captum
**รายวิชา:** จริยธรรมในการใช้ AI และ Explainable AI  
**เวลาโดยประมาณ:** 120 นาที | **Runtime:** แนะนำ GPU (Runtime → Change runtime type → T4 GPU) แต่ CPU ก็ทำงานได้

## วัตถุประสงค์
1. สร้าง attribution map ด้วย Saliency, SmoothGrad, Integrated Gradients (IG), Occlusion และ Grad-CAM
2. ตรวจสอบสัจพจน์ **completeness** ของ IG และผลของการเลือก baseline
3. ทำ **sanity check** แบบ cascading model randomization (Adebayo et al., 2018)
4. วัด **faithfulness** ด้วย deletion curve (Petsiuk et al., 2018) และเปรียบเทียบกับการเรียงลำดับแบบสุ่ม


In [ ]:
!pip -q install captum

In [ ]:
import copy, numpy as np, matplotlib.pyplot as plt, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from scipy.stats import spearmanr
from captum.attr import (Saliency, IntegratedGradients, NoiseTunnel, Occlusion,
                         LayerGradCam, LayerAttribution)

torch.manual_seed(0); np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## ส่วนที่ 1 ข้อมูล MNIST และ CNN ขนาดเล็ก
ใช้ `nn.ReLU()` แยกแต่ละชั้น (ไม่ใช้ inplace และไม่ใช้โมดูลซ้ำ) เพื่อให้วิธี attribution บางวิธีทำงานถูกต้อง

In [ ]:
MEAN, STD = 0.1307, 0.3081
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((MEAN,), (STD,))])
train_ds = datasets.MNIST("data", train=True, download=True, transform=tf)
test_ds = datasets.MNIST("data", train=False, download=True, transform=tf)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2)
test_dl = DataLoader(test_ds, batch_size=512)
BLACK = (0.0 - MEAN) / STD   # ค่าพิกเซลดำหลัง normalize ใช้เป็น baseline

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1); self.relu1 = nn.ReLU(); self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1); self.relu2 = nn.ReLU(); self.pool2 = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128); self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)
    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        return self.fc2(self.relu3(self.fc1(x.flatten(1))))

model = Net().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(2):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); loss = F.cross_entropy(model(xb), yb); loss.backward(); opt.step()
    model.eval(); correct = 0
    with torch.no_grad():
        for xb, yb in test_dl:
            correct += (model(xb.to(device)).argmax(1).cpu() == yb).sum().item()
    print(f"epoch {epoch+1}: test acc = {correct / len(test_ds):.4f}")

## ส่วนที่ 2 สร้าง attribution maps
- **Saliency:** $|\partial f_c / \partial x|$  
- **SmoothGrad:** ค่าเฉลี่ยของ gradient เมื่อเติม noise  
- **Integrated Gradients:** $IG_i(x) = (x_i - x'_i)\int_0^1 \frac{\partial f(x' + \alpha(x-x'))}{\partial x_i} d\alpha$  
- **Occlusion:** การเปลี่ยนแปลงของผลลัพธ์เมื่อบังบางส่วนของภาพ  
- **Grad-CAM:** $L^c = \mathrm{ReLU}(\sum_k \alpha_k^c A^k)$ โดย $\alpha_k^c$ = ค่าเฉลี่ยของ gradient บน feature map $A^k$

In [ ]:
model.eval()
xb, yb = next(iter(DataLoader(test_ds, batch_size=64, shuffle=True)))
x = xb[:6].to(device); pred = model(x).argmax(1)
base = torch.full_like(x, BLACK)

def attributions(m, x, target):
    out = {}
    out["Saliency"] = Saliency(m).attribute(x, target=target, abs=True)
    out["SmoothGrad"] = NoiseTunnel(Saliency(m)).attribute(x, nt_type="smoothgrad", nt_samples=25,
                                                           stdevs=0.3, target=target, abs=True)
    out["IntegratedGradients"] = IntegratedGradients(m).attribute(x, baselines=torch.full_like(x, BLACK),
                                                                 target=target, n_steps=64)
    out["Occlusion"] = Occlusion(m).attribute(x, target=target, sliding_window_shapes=(1, 6, 6),
                                              strides=(1, 2, 2), baselines=BLACK)
    gc = LayerGradCam(m, m.conv2).attribute(x, target=target)
    out["Grad-CAM"] = LayerAttribution.interpolate(gc, (28, 28), interpolate_mode="bilinear")
    return {k: v.detach() for k, v in out.items()}

A = attributions(model, x, pred)

def to_img(a):
    a = a.squeeze().abs().cpu().numpy()
    return a / (a.max() + 1e-12)

cols = ["Input"] + list(A.keys())
fig, axes = plt.subplots(len(x), len(cols), figsize=(2.2 * len(cols), 2.2 * len(x)))
for r in range(len(x)):
    axes[r, 0].imshow(x[r].squeeze().cpu() * STD + MEAN, cmap="gray")
    axes[r, 0].set_ylabel(f"pred={pred[r].item()}")
    for c, k in enumerate(A.keys(), start=1):
        axes[r, c].imshow(to_img(A[k][r]), cmap="inferno")
for c, name in enumerate(cols): axes[0, c].set_title(name, fontsize=9)
for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## ส่วนที่ 3 ตรวจสอบ completeness ของ IG และผลของ baseline
Completeness: $\sum_i IG_i(x) = f_c(x) - f_c(x')$ ค่า `delta` ควรเข้าใกล้ศูนย์เมื่อเพิ่ม `n_steps`

In [ ]:
ig = IntegratedGradients(model)
for steps in [8, 32, 128]:
    attr, delta = ig.attribute(x, baselines=base, target=pred, n_steps=steps, return_convergence_delta=True)
    print(f"n_steps={steps:4d}: max |delta| = {delta.abs().max().item():.5f}")

with torch.no_grad():
    fx = model(x).gather(1, pred[:, None]).squeeze(); fb = model(base).gather(1, pred[:, None]).squeeze()
print("Σ attributions :", attr.sum(dim=(1, 2, 3)).detach().cpu().numpy().round(3))
print("f(x) - f(x')    :", (fx - fb).cpu().numpy().round(3))

In [ ]:
baselines = {"black": torch.full_like(x, BLACK),
             "gaussian noise": BLACK + 0.5 * torch.randn_like(x),
             "blurred": F.avg_pool2d(x, 7, stride=1, padding=3)}
fig, axes = plt.subplots(1, len(baselines) + 1, figsize=(10, 2.8))
axes[0].imshow(x[0].squeeze().cpu() * STD + MEAN, cmap="gray"); axes[0].set_title("input")
for i, (name, b) in enumerate(baselines.items(), start=1):
    a = ig.attribute(x[:1], baselines=b[:1], target=pred[:1], n_steps=64)
    axes[i].imshow(to_img(a[0]), cmap="inferno"); axes[i].set_title(f"IG | {name}", fontsize=9)
for a in axes: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## ส่วนที่ 4 Sanity check: Cascading model randomization (Adebayo et al., 2018)
สุ่มค่าน้ำหนักทีละชั้นจากบนลงล่าง ถ้าคำอธิบาย *ไม่เปลี่ยน* แม้โมเดลถูกทำลาย แปลว่าวิธีนั้นไม่ได้สะท้อนสิ่งที่โมเดลเรียนรู้  
วัดความคล้ายด้วย Spearman rank correlation ของ |attribution|

In [ ]:
ORDER = ["fc2", "fc1", "conv2", "conv1"]

def randomized(m, layers):
    m2 = copy.deepcopy(m)
    for name in layers: getattr(m2, name).reset_parameters()
    return m2.eval()

def rank_sim(a, b):
    rs = [spearmanr(a[i].abs().flatten().cpu().numpy(), b[i].abs().flatten().cpu().numpy())[0]
          for i in range(a.shape[0])]
    return float(np.nanmean(rs))

methods = ["Saliency", "IntegratedGradients", "Grad-CAM"]
sim = {k: [1.0] for k in methods}
for k in range(1, len(ORDER) + 1):
    Ar = attributions(randomized(model, ORDER[:k]), x, pred)
    for mth in methods: sim[mth].append(rank_sim(A[mth], Ar[mth]))

labels = ["original"] + [f"+{l}" for l in ORDER]
for mth in methods: plt.plot(labels, sim[mth], "o-", label=mth)
plt.ylabel("Spearman rho vs. original attribution"); plt.title("Cascading randomization"); plt.legend(); plt.grid(alpha=.3); plt.show()

## ส่วนที่ 5 Faithfulness: Deletion curve
ลบพิกเซลที่สำคัญที่สุดก่อน (แทนด้วย baseline) แล้วติดตามความน่าจะเป็นของคลาสที่ทำนาย พื้นที่ใต้กราฟ **ยิ่งต่ำยิ่งดี** เทียบกับการลบแบบสุ่ม

In [ ]:
@torch.no_grad()
def deletion_curve(m, x1, target, attr1, steps=28):
    n = x1.numel(); order = attr1.abs().flatten().argsort(descending=True)
    batch = []
    for s in range(steps + 1):
        xm = x1.clone().flatten(); xm[order[: int(n * s / steps)]] = BLACK
        batch.append(xm.view_as(x1))
    probs = F.softmax(m(torch.stack(batch)), dim=1)[:, target]
    return probs.cpu().numpy()

xb2, _ = next(iter(DataLoader(test_ds, batch_size=50, shuffle=True)))
xb2 = xb2.to(device); pb2 = model(xb2).argmax(1)
A2 = attributions(model, xb2, pb2)
A2["Random"] = torch.rand_like(xb2)

curves = {k: np.mean([deletion_curve(model, xb2[i], pb2[i].item(), A2[k][i]) for i in range(len(xb2))], axis=0)
          for k in ["Saliency", "IntegratedGradients", "Occlusion", "Grad-CAM", "Random"]}
frac = np.linspace(0, 1, len(next(iter(curves.values()))))
for k, c in curves.items(): plt.plot(frac, c, label=f"{k} (AUC={c.mean():.3f})")
plt.xlabel("Fraction of pixels removed"); plt.ylabel("P(predicted class)"); plt.legend(); plt.grid(alpha=.3); plt.show()

## แบบฝึกหัดสำหรับรายงาน
1. ทำ **data randomization test**: ฝึกโมเดลบน label ที่สลับแบบสุ่ม แล้วเปรียบเทียบ attribution กับโมเดลปกติ
2. เพิ่ม insertion curve และรายงานทั้ง deletion/insertion AUC พร้อมช่วงความเชื่อมั่นจาก 200 ภาพ
3. อภิปรายว่า deletion metric อาจเอนเอียงเพราะสร้างภาพที่อยู่นอกการแจกแจง (out-of-distribution) อย่างไร และเสนอทางแก้ (เช่น ROAR: remove-and-retrain)
4. เขียนข้อเสนอการใช้ saliency ในงานภาพทางการแพทย์ พร้อมข้อจำกัดทางจริยธรรมที่ต้องแจ้งผู้ใช้
